# Programming homework assignment Chapter 2 - Agents
**Course:** ICOM 5015-001D Artificial Intelligence  
**Student:** Jesús Y. Cabán Feliciano, Victor A Gerena-Hilerio,Nollan N Rivera-Febus, Alejandro A Roberts-Quintana

This notebook presents the implementation and analysis for Exercises 2.11 and 2.14 from Russell and Norvig’s vacuum world problems. The simulator and agents were implemented in Python using a modular design that separates the agent program, environment dynamics, percept generation, and performance evaluation.

## Introduction

This homework studies two versions of the vacuum-cleaner world.

- **Exercise 2.11** requires implementing a performance-measuring simulator for the vacuum world using the **PEAS framework**.
- **Exercise 2.14** studies a more difficult version of the environment in which the geography, boundaries, obstacles, and initial dirt placement are unknown.

The implementation was written in Python and designed to be modular. In both exercises, the code separates:

1. **Agent program** – maps percepts to actions  
2. **Environment state and dynamics** – stores and updates the world  
3. **Performance evaluator** – computes cumulative score  

This separation ensures that the agent does not directly modify the environment and that the environment does not depend on any single agent implementation.

## Exercise 2.11 – Performance-Measuring Vacuum World

### Problem Statement
This exercise asks for a modular implementation of the vacuum-cleaner world simulator, explicitly defined as a **PEAS task environment**, with configurable sensors, actuators, and environment characteristics.

---

### PEAS Specification

#### **P — Performance Measure**
The simulator uses the following quantitative performance function:

- **+1 for each clean square at every time step**
- **−1 for each action executed**

If there are \( C \) clean squares at a time step, then the score update is:

\[
\Delta P = C - 1
\]

The cumulative score is tracked during the simulation and reported at the end of each run.

This performance measure rewards the agent for cleaning the environment quickly and for keeping squares clean over time, while penalizing unnecessary actions.

---

#### **E — Environment**
The environment is explicitly represented by a `VacuumState` object containing:

- the set of locations,
- the adjacency graph that defines spatial structure,
- the current location of the agent,
- the dirt distribution,
- and the current simulation step.

The minimum required world is the classic **two-location world** with locations **A** and **B**, but the code is extensible to larger environments such as line worlds or other graph-based layouts.

The environment evolves independently of the agent through the transition function implemented in `VacuumEnvironment.step()`.

---

#### **A — Actuators**
The action set includes:

- `Left`
- `Right`
- `Suck`
- `NoOp`

These actions are implemented through a modular dictionary of action handlers, which makes it easy to extend the simulator with additional actions later.

---

#### **S — Sensors**
The percept delivered to the agent includes:

- the current location,
- whether the current square is dirty.

This is implemented using a modular sensor interface. The code includes both:

- `LocalDirtSensor`
- `NoisyLocalDirtSensor`

Therefore, alternative sensor models can be substituted without redesigning the simulator.

---

### Architectural Design
The simulator clearly separates:

- **Agent program**: maps percepts to actions
- **Environment state and dynamics**: applies actions and updates the world
- **Performance evaluator**: computes and accumulates the score

This satisfies the architectural requirement that:

- the agent must not directly modify the environment state,
- and the environment must not assume a specific agent implementation.

---

### Agent Programs Tested

#### **SimpleReflexAgent**
This agent follows a deterministic rule:

- if the current square is dirty, execute `Suck`
- otherwise, execute `Right`

#### **RandomAgent**
This agent selects uniformly at random among:

- `Suck`
- `Left`
- `Right`
- `NoOp`

---

### Experimental Setup
To satisfy the experimental requirement, the simulator was run with at least two different agent programs.

For the comparison:

- **100 trials** were executed
- each trial lasted **30 steps**
- the initial dirt distribution and starting location were randomized

Each trial used a fresh environment, fresh evaluator, and fresh agent instance.

---

### Results

Single demonstration run:

- Final state: both locations clean
- Cumulative performance: **10**

Experimental comparison over 100 trials, 30 steps each:

- **SimpleReflexAgent:** mean = **29.29**, stdev = **0.83**
- **RandomAgent:** mean = **21.52**, stdev = **8.55**

---

### Discussion
The simple reflex agent performs better than the random agent because the two-location vacuum world is extremely small and the percept provides enough information for effective local behavior. If the square is dirty, the agent cleans it; otherwise, moving right is usually enough to reach the other square.

The random agent performs worse because it often wastes actions by choosing `NoOp`, moving unnecessarily, or trying to suck clean squares. Since the performance measure penalizes every action, these wasted steps reduce the total score. This also explains the much larger standard deviation for the random agent.

In PEAS terms:

- the **sensor** gives enough local information for useful reflex behavior,
- the **actuators** are sufficient to clean the world,
- the **performance measure** rewards quick and efficient cleaning,
- and the **environment** is simple enough that a deterministic reflex policy works well.

Thus, in this environment, the simple reflex agent is better matched to the task than the random agent.

In [1]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Dict, List, Protocol, Callable, Optional, Tuple
import random
import statistics

"""
PEAS specification for the vacuum world simulator:

P (Performance measure): cumulative score where each time step the
    environment gives +1 per clean square and subtracts 1 per action.
    The simulator tracks `cumulative_score` in `PerformanceEvaluator`.

E (Environment): explicit `VacuumEnvironment` with a `VacuumState` that
    contains locations (graph adjacency), dirt distribution, agent location,
    and step counter. The environment evolves via its transition (action
    handlers) and does not depend on the agent implementation.

A (Actuators): modular `action_handlers` mapping action names (e.g.,
    Left/Right/Suck/NoOp) to handler callables. New actions can be added by
    adding entries to this mapping.

S (Sensors): modular `SensorModel` protocol. Example implementations include
    `LocalDirtSensor` and `NoisyLocalDirtSensor`. Percepts are produced by
    calling `env.percept()` which delegates to the sensor.
"""



# ============================================================
# Actuators (Actions) - extensible
# ============================================================

class Action:
    LEFT = "Left"
    RIGHT = "Right"
    SUCK = "Suck"
    NOOP = "NoOp"


# ============================================================
# Sensors (Percepts) - swappable / modular
# ============================================================

@dataclass(frozen=True)
class Percept:
    location: str
    is_dirty: bool


class SensorModel(Protocol):
    def sense(self, env: "VacuumEnvironment") -> Percept:
        ...


class LocalDirtSensor:
    """Percept = (current location, dirt status at current location)."""
    def sense(self, env: "VacuumEnvironment") -> Percept:
        loc = env.state.agent_location
        return Percept(location=loc, is_dirty=env.state.dirt.get(loc, False))


class NoisyLocalDirtSensor:
    """Example alternative sensor: flips dirt reading with probability flip_prob."""
    def __init__(self, flip_prob: float = 0.1):
        self.flip_prob = flip_prob

    def sense(self, env: "VacuumEnvironment") -> Percept:
        loc = env.state.agent_location
        true_dirty = env.state.dirt.get(loc, False)
        observed_dirty = (not true_dirty) if random.random() < self.flip_prob else true_dirty
        return Percept(location=loc, is_dirty=observed_dirty)


# ============================================================
# Environment State (size/shape extensible)
# ============================================================

@dataclass
class VacuumState:
    locations: List[str]                 # e.g., ["A","B"] or more
    adjacency: Dict[str, List[str]]      # shape as a graph
    agent_location: str
    dirt: Dict[str, bool]                # dirt distribution
    step: int = 0


# ============================================================
# Performance Measure (tracks cumulative performance)
# ============================================================

class PerformanceEvaluator:
    """
    Example performance function from the textbook-style description:
      +1 per clean square per time step
      -1 per action per time step
    Easily adjustable.
    """

    def __init__(self, reward_clean_per_square: int = 1, action_cost: int = 1):
        self.reward_clean_per_square = reward_clean_per_square
        self.action_cost = action_cost
        self.cumulative_score = 0

    def score_step(self, state: VacuumState, action: str) -> int:
        clean_squares = sum(1 for loc in state.locations if not state.dirt.get(loc, False))
        delta = (self.reward_clean_per_square * clean_squares) - self.action_cost
        self.cumulative_score += delta
        return delta


# ============================================================
# Environment Dynamics (transition function) - independent
# ============================================================

class VacuumEnvironment:
    """
    Modular, performance-measuring simulator for the vacuum world.

    Key property: agent program is separate.
    The environment provides percepts and applies actions via a transition function.
    """

    def __init__(
        self,
        initial_state: VacuumState,
        sensor: SensorModel,
        evaluator: PerformanceEvaluator,
        action_handlers: Optional[Dict[str, Callable[["VacuumEnvironment"], None]]] = None,
    ):
        self.state = initial_state
        self.sensor = sensor
        self.evaluator = evaluator

        # Modular actuators: action -> handler mapping (easy to extend)
        self.action_handlers = action_handlers or {
            Action.LEFT: self._handle_left,
            Action.RIGHT: self._handle_right,
            Action.SUCK: self._handle_suck,
            Action.NOOP: self._handle_noop,
        }

    def percept(self) -> Percept:
        """Generate a percept for the agent (modular sensor model)."""
        return self.sensor.sense(self)

    def step(self, action: str) -> Tuple[VacuumState, int]:
        """
        Apply one action using the environment transition function.
        Returns (new_state, performance_delta_for_this_step).
        """
        if action not in self.action_handlers:
            raise ValueError(f"Unknown action: {action}")

        # Apply action (environment changes itself)
        self.action_handlers[action](self)

        # Advance time
        self.state.step += 1

        # Update performance (environment tracks it)
        delta = self.evaluator.score_step(self.state, action)
        return self.state, delta

    def run(self, agent_program: "AgentProgram", steps: int) -> int:
        """
        Run a simulation episode for a fixed number of steps.
        Returns cumulative performance score.
        """
        for _ in range(steps):
            p = self.percept()
            a = agent_program(p)      # agent maps percept -> action
            self.step(a)              # environment applies action
        return self.evaluator.cumulative_score

    # ---------- Default action handlers ----------
    # Written to work for any adjacency graph.

    def _handle_left(self, _: "VacuumEnvironment") -> None:
        neighbors = self.state.adjacency.get(self.state.agent_location, [])
        if neighbors:
            self.state.agent_location = neighbors[0]

    def _handle_right(self, _: "VacuumEnvironment") -> None:
        neighbors = self.state.adjacency.get(self.state.agent_location, [])
        if neighbors:
            self.state.agent_location = neighbors[-1]

    def _handle_suck(self, _: "VacuumEnvironment") -> None:
        self.state.dirt[self.state.agent_location] = False

    def _handle_noop(self, _: "VacuumEnvironment") -> None:
        pass


# ============================================================
# Agent program interface (kept separate from environment)
# ============================================================

class AgentProgram(Protocol):
    def __call__(self, percept: Percept) -> str:
        ...


# Simple demo agent (optional): classic reflex
class SimpleReflexAgent:
    def __call__(self, percept: Percept) -> str:
        if percept.is_dirty:
            return Action.SUCK
        return Action.RIGHT


# Random agent: chooses uniformly among available primitive actions.
class RandomAgent:
    def __init__(self, seed: Optional[int] = None):
        self.rng = random.Random(seed)

    def __call__(self, percept: Percept) -> str:
        return self.rng.choice([Action.SUCK, Action.LEFT, Action.RIGHT, Action.NOOP])


# ============================================================
# World builder utilities (size/shape/dirt placement configurable)
# ============================================================

def build_two_location_world(
    dirt_A: bool = True,
    dirt_B: bool = True,
    start: str = "A",
) -> VacuumState:
    locations = ["A", "B"]
    adjacency = {"A": ["B"], "B": ["A"]}  # A <-> B
    dirt = {"A": dirt_A, "B": dirt_B}
    return VacuumState(locations=locations, adjacency=adjacency, agent_location=start, dirt=dirt)


def build_line_world(n: int, dirt_prob: float = 0.5, start_index: int = 0, seed: int = 0) -> VacuumState:
    """
    Example extensibility: a line of n locations (L0-L{n-1}).
    'Left' moves to previous neighbor, 'Right' moves to next neighbor.
    Dirt placement is random with probability dirt_prob.
    """
    rng = random.Random(seed)
    locations = [f"L{i}" for i in range(n)]
    adjacency: Dict[str, List[str]] = {}
    for i, loc in enumerate(locations):
        neighbors = []
        if i - 1 >= 0:
            neighbors.append(locations[i - 1])
        if i + 1 < n:
            neighbors.append(locations[i + 1])
        adjacency[loc] = neighbors

    dirt = {loc: (rng.random() < dirt_prob) for loc in locations}
    start = locations[max(0, min(start_index, n - 1))]
    return VacuumState(locations=locations, adjacency=adjacency, agent_location=start, dirt=dirt)


# ============================================================
# Demo main (shows it runs + prints cumulative performance)
# ============================================================

def main() -> None:
    # Minimum required world: two locations
    state = build_two_location_world(dirt_A=True, dirt_B=False, start="A")

    # Modular components:
    sensor = LocalDirtSensor()  # swap to NoisyLocalDirtSensor(...) if you want
    evaluator = PerformanceEvaluator(reward_clean_per_square=1, action_cost=1)

    env = VacuumEnvironment(initial_state=state, sensor=sensor, evaluator=evaluator)

    # Demo run with a simple reflex agent
    agent = SimpleReflexAgent()
    total = env.run(agent, steps=10)

    print("=== Vacuum World Simulator (Exercise 2.11) ===")
    print("Final state:", env.state)
    print("Cumulative performance (simple reflex, single run):", total)

    # Experimental requirement: run at least two different agents (reflex vs random)
    def run_trials(agent_factory: Callable[[], AgentProgram], trials: int = 100, steps: int = 30):
        results = []
        for t in range(trials):
            # Randomize initial dirt distribution for each trial (two-location world)
            dirtA = random.choice([True, False])
            dirtB = random.choice([True, False])
            state_t = build_two_location_world(dirt_A=dirtA, dirt_B=dirtB, start=random.choice(["A", "B"]))
            sensor_t = LocalDirtSensor()
            evaluator_t = PerformanceEvaluator(reward_clean_per_square=1, action_cost=1)
            env_t = VacuumEnvironment(initial_state=state_t, sensor=sensor_t, evaluator=evaluator_t)
            agent_t = agent_factory()
            score = env_t.run(agent_t, steps=steps)
            results.append(score)
        return results

    trials = 100
    steps = 30
    reflex_results = run_trials(lambda: SimpleReflexAgent(), trials=trials, steps=steps)
    random_results = run_trials(lambda: RandomAgent(), trials=trials, steps=steps)

    print(f"\nExperimental comparison over {trials} trials, {steps} steps each:")
    print(f"SimpleReflexAgent: mean={statistics.mean(reflex_results):.2f}, stdev={statistics.pstdev(reflex_results):.2f}")
    print(f"RandomAgent:      mean={statistics.mean(random_results):.2f}, stdev={statistics.pstdev(random_results):.2f}")

if __name__ == "__main__":
    main()

=== Vacuum World Simulator (Exercise 2.11) ===
Final state: VacuumState(locations=['A', 'B'], adjacency={'A': ['B'], 'B': ['A']}, agent_location='B', dirt={'A': False, 'B': False}, step=10)
Cumulative performance (simple reflex, single run): 10

Experimental comparison over 100 trials, 30 steps each:
SimpleReflexAgent: mean=29.29, stdev=0.83
RandomAgent:      mean=21.52, stdev=8.55


## Exercise 2.14 – Vacuum World with Unknown Geography

### Problem Statement
In this modified version of the vacuum environment, the agent does not know:

- the extent of the environment,
- its boundaries,
- the obstacle locations,
- or the initial dirt configuration.

The agent can move in four directions:

- `Up`
- `Down`
- `Left`
- `Right`

and may also execute:

- `Suck`
- `NoOp`

The percept includes:

- the current location,
- whether the current square is dirty,
- whether the previous movement caused a bump.

This environment is much harder because the agent must operate under incomplete knowledge of the world.

---

### 1. Can a simple reflex agent be perfectly rational?

No, a simple reflex agent cannot be perfectly rational in this environment.

A simple reflex agent chooses an action using only the current percept. However, in this problem, the agent does not know the full environment map or where remaining dirt may be located. Two different global situations can produce exactly the same local percept, even though the correct action may be different.

For example, the current square may be clean and the agent may not have bumped, but that does not tell the agent whether it should go left, right, up, or down to reach dirty squares. Because the environment is partially observable and initially unknown, perfect rationality requires memory or an internal model. A simple reflex agent has neither, so it cannot always behave optimally.

---

### 2. Can a randomized simple reflex agent outperform a simple reflex agent?

Yes.

The notebook implements a **RandomizedReflexAgent** with the following policy:

- if the current square is dirty, execute `Suck`
- otherwise, choose randomly among the movement actions and `NoOp`

This agent can outperform the deterministic simple reflex agent because the deterministic agent always moves in one fixed direction when the square is clean. In an unknown two-dimensional environment with obstacles, such a rigid rule leads to poor exploration.

The randomized agent explores more of the environment and therefore discovers more dirty locations.

#### Results on Random Environments
Over **40 random environments** with **200 steps each**:

- **SimpleReflex:** mean = **5734.00**, stdev = **504.60**
- **RandomizedReflex:** mean = **6101.30**, stdev = **384.34**
- **StatefulReflex:** mean = **6098.57**, stdev = **455.79**

These results show that the randomized reflex agent outperformed the deterministic simple reflex agent on average.

---

### 3. Can an environment be designed where the randomized agent performs poorly?

Yes.

A randomized agent performs poorly when the environment rewards directed, consistent motion rather than broad exploration.

To demonstrate this, the notebook includes a **corridor experiment**:

- a 1-by-\( n \) environment,
- the agent starts at the far left,
- and the only dirt is located at the far right end.

In this case, random actions cause unnecessary wandering and delays, while a deterministic strategy that repeatedly moves right is actually more effective.

#### Corridor Results
For a corridor of length 12 and 300 steps:

- **RandomizedReflex:** score = **3234**
- **StatefulReflex:** score = **3266**
- **SimpleReflex:** score = **3289**

In this environment, the randomized reflex agent performed the worst.

This shows that randomness helps in unknown open environments, but it can hurt performance in highly structured environments.

---

### 4. Can a reflex agent with state outperform a simple reflex agent?

Yes.

The notebook implements a **StatefulReflexAgent** that maintains internal state, including:

- visited cells,
- known free cells,
- blocked directions,
- previous location,
- previous action.

Using bump feedback, the agent learns which directions are blocked from particular cells. It also prefers unvisited neighbors, which improves exploration of unknown environments.

#### Measured Results
Over the same 40 random environments:

- **SimpleReflex:** mean = **5734.00**
- **StatefulReflex:** mean = **6098.57**

Therefore, the reflex agent with state outperformed the simple reflex agent.

---

### Can a rational agent of this type be designed?

A reflex agent with state can be made more rational than a simple reflex agent, but it is still not fully rational unless it maintains a richer internal model and uses planning.

The implemented stateful agent is better than the simple reflex agent because it remembers visited locations and blocked moves. However, it still does not:

- construct a complete map,
- compute optimal paths,
- or deliberately plan routes toward unexplored or dirty regions.

A more rational design would maintain an internal map of discovered cells and explicitly plan actions toward the nearest dirty or unexplored location. Therefore, the stateful reflex agent is an improvement, but not a perfectly rational solution.

---

### Discussion
The results illustrate an important AI principle: as environments become less observable and more complex, immediate reflex rules become less effective.

- The **simple reflex agent** is too rigid for unknown geography.
- The **randomized reflex agent** benefits from better exploration.
- The **stateful reflex agent** uses memory to avoid repeating mistakes and to explore more systematically.

The best architecture depends on the structure of the environment. Randomness helps in open unknown spaces, while memory helps in partially observable environments where past experience matters.

In [2]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Dict, Tuple, Set, Optional, Protocol, List, Callable
import random
import statistics


# ============================================================
# Actions
# ============================================================

class Action:
    UP = "Up"
    DOWN = "Down"
    LEFT = "Left"
    RIGHT = "Right"
    SUCK = "Suck"
    NOOP = "NoOp"


# ============================================================
# Percepts 
# ============================================================

@dataclass(frozen=True)
class Percept:
    location: Tuple[int, int]
    is_dirty: bool
    bump: bool  # True if last movement action failed (wall/obstacle)


class AgentProgram(Protocol):
    def __call__(self, percept: Percept) -> str:
        ...


# ============================================================
# Environment state
# ============================================================

@dataclass
class GridState:
    width: int
    height: int
    obstacles: Set[Tuple[int, int]]
    dirt: Dict[Tuple[int, int], bool]
    agent_location: Tuple[int, int]
    step: int = 0


# ============================================================
# Performance measure
# ============================================================

class PerformanceEvaluator:
    def __init__(self, reward_clean_per_square: int = 1, action_cost: int = 1):
        self.reward_clean_per_square = reward_clean_per_square
        self.action_cost = action_cost
        self.cumulative_score = 0

    def score_step(self, state: GridState, action: str) -> int:
        clean_squares = sum(
            1
            for x in range(state.width)
            for y in range(state.height)
            if (x, y) not in state.obstacles and not state.dirt.get((x, y), False)
        )
        delta = (self.reward_clean_per_square * clean_squares) - self.action_cost
        self.cumulative_score += delta
        return delta


# ============================================================
# Environment dynamics
# ============================================================

class GridEnvironment:
    """Grid environment with unknown geography to the agent."""

    def __init__(self, initial_state: GridState, evaluator: PerformanceEvaluator):
        self.state = initial_state
        self.evaluator = evaluator
        self.last_bump: bool = False

    def percept(self) -> Percept:
        loc = self.state.agent_location
        return Percept(
            location=loc,
            is_dirty=self.state.dirt.get(loc, False),
            bump=self.last_bump,
        )

    def step(self, action: str) -> Tuple[GridState, int]:
        x, y = self.state.agent_location
        new_loc = (x, y)
        self.last_bump = False  # reset each step

        if action == Action.UP:
            candidate = (x, y - 1)
            if self._is_free(candidate):
                new_loc = candidate
            else:
                self.last_bump = True
        elif action == Action.DOWN:
            candidate = (x, y + 1)
            if self._is_free(candidate):
                new_loc = candidate
            else:
                self.last_bump = True
        elif action == Action.LEFT:
            candidate = (x - 1, y)
            if self._is_free(candidate):
                new_loc = candidate
            else:
                self.last_bump = True
        elif action == Action.RIGHT:
            candidate = (x + 1, y)
            if self._is_free(candidate):
                new_loc = candidate
            else:
                self.last_bump = True
        elif action == Action.SUCK:
            self.state.dirt[self.state.agent_location] = False
        elif action == Action.NOOP:
            pass
        else:
            raise ValueError(f"Unknown action: {action}")

        self.state.agent_location = new_loc
        self.state.step += 1
        delta = self.evaluator.score_step(self.state, action)
        return self.state, delta

    def _is_free(self, loc: Tuple[int, int]) -> bool:
        x, y = loc
        if x < 0 or y < 0 or x >= self.state.width or y >= self.state.height:
            return False
        if loc in self.state.obstacles:
            return False
        return True


# ============================================================
# World builder
# ============================================================

def build_random_grid(
    width: int,
    height: int,
    obstacle_prob: float,
    dirt_prob: float,
    seed: Optional[int] = None
) -> GridState:
    rng = random.Random(seed)
    obstacles: Set[Tuple[int, int]] = set()
    dirt: Dict[Tuple[int, int], bool] = {}

    for x in range(width):
        for y in range(height):
            if rng.random() < obstacle_prob:
                obstacles.add((x, y))
            else:
                dirt[(x, y)] = (rng.random() < dirt_prob)

    free_cells = [c for c in dirt.keys() if c not in obstacles]
    if not free_cells:
        raise ValueError("No free cells available")

    start = rng.choice(free_cells)
    return GridState(
        width=width,
        height=height,
        obstacles=obstacles,
        dirt=dirt,
        agent_location=start,
    )


# ============================================================
# Agents
# ============================================================

class SimpleReflexAgent:
    """Simple reflex: if dirty -> SUCK else always move RIGHT (blind)."""
    def __call__(self, percept: Percept) -> str:
        if percept.is_dirty:
            return Action.SUCK
        return Action.RIGHT


class RandomizedReflexAgent:
    """If dirty -> SUCK else pick a random move."""
    def __init__(self, seed: Optional[int] = None):
        self.rng = random.Random(seed)

    def __call__(self, percept: Percept) -> str:
        if percept.is_dirty:
            return Action.SUCK
        return self.rng.choice([Action.UP, Action.DOWN, Action.LEFT, Action.RIGHT, Action.NOOP])


class StatefulReflexAgent:
    """
    Reflex agent with state:
    - Uses bump feedback to learn blocked directions from specific cells.
    - Tracks visited cells and prefers unvisited neighbors first.
    """

    DELTAS = {
        Action.UP: (0, -1),
        Action.DOWN: (0, 1),
        Action.LEFT: (-1, 0),
        Action.RIGHT: (1, 0),
    }

    def __init__(self):
        self.known_free: Set[Tuple[int, int]] = set()
        self.blocked: Dict[Tuple[int, int], Set[str]] = {}
        self.visited: Set[Tuple[int, int]] = set()

        self.last_location: Optional[Tuple[int, int]] = None
        self.last_action: Optional[str] = None

    def __call__(self, percept: Percept) -> str:
        loc = percept.location
        self.known_free.add(loc)
        self.visited.add(loc)

        # Learns blocked transitions from bump feedback.
        if percept.bump and self.last_location is not None and self.last_action in self.DELTAS:
            self.blocked.setdefault(self.last_location, set()).add(self.last_action)

        if percept.is_dirty:
            self.last_location = loc
            self.last_action = Action.SUCK
            return Action.SUCK

        # Prefers unvisited neighbors
        for a, (dx, dy) in self.DELTAS.items():
            if a in self.blocked.get(loc, set()):
                continue
            cand = (loc[0] + dx, loc[1] + dy)
            if cand not in self.visited:
                self.last_location = loc
                self.last_action = a
                return a

        # Otherwise, try any unblocked move
        for a in self.DELTAS:
            if a in self.blocked.get(loc, set()):
                continue
            self.last_location = loc
            self.last_action = a
            return a

        self.last_location = loc
        self.last_action = Action.NOOP
        return Action.NOOP


# ============================================================
# Experiments
# ============================================================

def run_episode(env: GridEnvironment, agent: AgentProgram, steps: int) -> int:
    for _ in range(steps):
        p = env.percept()
        a = agent(p)
        env.step(a)
    return env.evaluator.cumulative_score


def experiment_random_environments(
    num_envs: int = 50,
    width: int = 6,
    height: int = 6,
    obstacle_prob: float = 0.1,
    dirt_prob: float = 0.2,
    steps: int = 200,
):
    rng = random.Random(0)

    # use factories to create a fresh agent per environment/run
    agent_factories: List[Tuple[str, Callable[[], AgentProgram]]] = [
        ("SimpleReflex", lambda: SimpleReflexAgent()),
        ("RandomizedReflex", lambda: RandomizedReflexAgent()),
        ("StatefulReflex", lambda: StatefulReflexAgent()),
    ]

    results: Dict[str, List[float]] = {name: [] for name, _ in agent_factories}

    for _ in range(num_envs):
        state = build_random_grid(width, height, obstacle_prob, dirt_prob, seed=rng.randint(0, 10**9))

        for name, make_agent in agent_factories:
            evaluator = PerformanceEvaluator(reward_clean_per_square=1, action_cost=1)

            # Copy state so each agent faces the same environment
            state_copy = GridState(
                width=state.width,
                height=state.height,
                obstacles=set(state.obstacles),
                dirt=dict(state.dirt),
                agent_location=state.agent_location,
            )
            env = GridEnvironment(initial_state=state_copy, evaluator=evaluator)
            agent = make_agent()  # fresh instance each run

            score = run_episode(env, agent, steps)
            results[name].append(score)

    print(f"\nExperiment: {num_envs} random environments, {steps} steps each")
    for name, vals in results.items():
        print(f"{name}: mean={statistics.mean(vals):.2f}, stdev={statistics.pstdev(vals):.2f}")


def experiment_corridor_poor_for_random(length: int = 10, dirt_at_end: bool = True, steps: int = 200):
    # 1 x length corridor: start at (0,0) and dirt at far right
    width = length
    height = 1
    obstacles: Set[Tuple[int, int]] = set()
    dirt: Dict[Tuple[int, int], bool] = {(x, 0): False for x in range(width)}
    if dirt_at_end:
        dirt[(width - 1, 0)] = True

    state = GridState(width=width, height=height, obstacles=obstacles, dirt=dirt, agent_location=(0, 0))

    agents: List[Tuple[str, Callable[[], AgentProgram]]] = [
        ("RandomizedReflex", lambda: RandomizedReflexAgent(seed=1)),
        ("StatefulReflex", lambda: StatefulReflexAgent()),
        ("SimpleReflex", lambda: SimpleReflexAgent()),
    ]

    print(f"\nCorridor experiment length={length}, dirt_at_end={dirt_at_end}, steps={steps}")
    for name, make_agent in agents:
        evaluator = PerformanceEvaluator(reward_clean_per_square=1, action_cost=1)
        state_copy = GridState(
            width=state.width,
            height=state.height,
            obstacles=set(state.obstacles),
            dirt=dict(state.dirt),
            agent_location=state.agent_location,
        )
        env = GridEnvironment(initial_state=state_copy, evaluator=evaluator)
        agent = make_agent()
        score = run_episode(env, agent, steps)
        print(f"{name}: score={score}")


def main():
    print("=== Vacuum World (Exercise 2.14) experiments ===")
    experiment_random_environments(
        num_envs=40,
        width=6,
        height=6,
        obstacle_prob=0.08,
        dirt_prob=0.12,
        steps=200,
    )
    experiment_corridor_poor_for_random(length=12, dirt_at_end=True, steps=300)


if __name__ == "__main__":
    main()

=== Vacuum World (Exercise 2.14) experiments ===

Experiment: 40 random environments, 200 steps each
SimpleReflex: mean=5734.00, stdev=504.60
RandomizedReflex: mean=6101.30, stdev=384.34
StatefulReflex: mean=6098.57, stdev=455.79

Corridor experiment length=12, dirt_at_end=True, steps=300
RandomizedReflex: score=3234
StatefulReflex: score=3266
SimpleReflex: score=3289


## Conclusion

This homework demonstrates how the design of an agent must match the structure of its environment.

In **Exercise 2.11**, the classic two-location vacuum world is simple enough that a deterministic simple reflex agent performs very well. The modular PEAS-based implementation successfully separates the agent, sensors, actuators, environment dynamics, and performance evaluator.

In **Exercise 2.14**, the problem becomes harder because the environment is unknown and only partially observable. In this setting:

- a simple reflex agent is not perfectly rational,
- a randomized reflex agent can outperform a deterministic one by improving exploration,
- but randomness can also perform poorly in structured environments such as corridors,
- and a reflex agent with state performs better because memory helps guide future actions.

Overall, the experiments show that as the environment becomes more complex and less observable, effective agents require more than immediate condition-action rules. Exploration and internal state become increasingly important.